# Compare Different Models on Real-World Datasets

Evaluate all registered models on OpenML benchmark datasets, with per-model cache reuse and W&B logging.

In [ ]:
from pathlib import Path
from typing import Any

from pfns.run_evaluation_cli import (
    build_available_baseline_model_configs,
    compute_per_dataset_stats,
    print_results_summary,
    run_real_world_model_from_config,
    summarize_results,
)
from pfns.experiments.model_benchmarks.hashing import single_model_hash
from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.io import (
    REAL_WORLD_BUNDLE_KEYS,
    REAL_WORLD_REQUIRED_FILES,
    download_results_bundle_from_wandb,
    find_latest_real_world_bundle_for_model,
    load_dataframe_bundle,
    make_bundle_path,
    make_model_artifact_name,
    
    sanitize_wandb_artifact_component,
    save_dataframe_bundle,
    upload_results_bundle_to_wandb,
)
from pfns.experiments.model_benchmarks.model_registry import (
    MODEL_FAMILIES,
    get_all_models,
    get_baseline_models,
    get_models_from_families,
    get_models_from_names,
)
from pfns.experiments.model_benchmarks.path_utils import build_repo_output_root
from pfns.experiments.model_benchmarks.real_world_presets import (
    get_real_world_preset,
)
from pfns.experiments.model_benchmarks.workflows import (
    aggregate_real_world_results_from_bundles,
    alias_real_world_dataframe_bundle,
    build_real_world_run_metadata,
    real_world_bundle_is_compatible,
)

REAL_WORLD_PRESET = "tabarena" # "openml"  # or "tabarena"
preset_defaults = get_real_world_preset(REAL_WORLD_PRESET)
REAL_DATASET_EXPERIMENT: dict[str, Any] = preset_defaults["experiment"]

BASELINE_EVAL: dict[str, int] = {
    "n_jobs": 4,
    "random_state": 42,
}

WANDB: dict[str, Any] = {
    "enabled": True,
    "overwrite": False,
    "artifact_name_real_eval": "real_eval_results",
    "entity": "icl_arch",
    "mode": "online",
} | preset_defaults["wandb"]

DATASET_SIZE_BUCKET_Q = 5

OUTPUT_ROOT = build_repo_output_root(Path.cwd(), "real_world_eval")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Results are stored in: {OUTPUT_ROOT}")
print(f"Available model families: {list(MODEL_FAMILIES)}")
print(f"Preset: {REAL_WORLD_PRESET}")


def get_model_combinations(*model_dicts, add_baselines=False):
    combined = {}
    if add_baselines:
        baseline_models_to_compare, skipped_baselines = build_available_baseline_model_configs(
            candidates=get_baseline_models(),
            n_jobs=BASELINE_EVAL["n_jobs"],
            random_state=BASELINE_EVAL["random_state"],
        )
        if skipped_baselines:
            print(f"Skipping unavailable baselines in this environment: {skipped_baselines}")
        combined.update(baseline_models_to_compare)
        
    if not model_dicts:
        return {}
    
    for d in model_dicts:
        combined.update(d)
    return combined

# Model selection

# All models
# models_to_compare = get_model_combinations(get_all_models(), add_baselines=True)

# All models without baselines
# models_to_compare = get_model_combinations(get_all_models(), add_baselines=False)

# Only baselines
# models_to_compare = get_model_combinations( {}, add_baselines=True)

# Equal param models
# models_to_compare = get_model_combinations(get_models_from_families(["equal_params"]), add_baselines=True)

# Transformer masking expreiments
# models_to_compare = get_model_combinations(get_models_from_families(["transformer_masked"]), add_baselines=False)

# All FLA models
# models_to_compare = get_model_combinations(get_models_from_families(["fla_models"]), add_baselines=False)

models_to_compare = get_model_combinations(get_models_from_families(["mimetic_initialization"]), add_baselines=False)
# models_to_compare = get_models_from_names(["Softmax_Transformer"])
# models_to_compare.update(get_models_from_names(["equal_params:Transformer_Comb_ST"]))


print(
    "Selected models:", len(models_to_compare)
)

device = get_default_device()
print(f"Using device: {device}")

expected_real_metadata = build_real_world_run_metadata(
    experiment=REAL_DATASET_EXPERIMENT,
    device=device,
)


In [ ]:
if WANDB["enabled"] and WANDB["overwrite"]:
    print("WANDB overwrite=True: skipping per-model download and forcing reruns.")

for model_name, model_config in models_to_compare.items():
    model_hash = single_model_hash(
        model_name=model_name,
        model_config=model_config,
        experiment_payload=REAL_DATASET_EXPERIMENT,
    )
    model_artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name_real_eval"],
        model_name=model_name,
        model_hash=model_hash,
    )

    if WANDB["enabled"] and not WANDB["overwrite"]:
        cached_bundle_path = download_results_bundle_from_wandb(
            artifact_name=model_artifact_name,
            download_root=OUTPUT_ROOT / "wandb_model_cache" / "real_world",
            required_files=REAL_WORLD_REQUIRED_FILES,
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
        )
        print(f"Checked for cached W&B artifact for {model_name}: {cached_bundle_path}")
        if cached_bundle_path is not None:
            cached_bundle = load_dataframe_bundle(
                cached_bundle_path,
                expected_keys=REAL_WORLD_BUNDLE_KEYS,
            )
            cached_bundle_for_model, source_labels = alias_real_world_dataframe_bundle(
                cached_bundle,
                target_model_name=model_name,
            )
            cached_dataframes = cached_bundle_for_model["dataframes"]

            if real_world_bundle_is_compatible(
                cached_bundle_for_model,
                model_name=model_name,
                expected_metadata=expected_real_metadata,
            ):
                model_bundle_path = make_bundle_path(
                    OUTPUT_ROOT / "real_world",
                    f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
                )
                save_dataframe_bundle(
                    dataframes=cached_dataframes,
                    bundle_dir=model_bundle_path,
                    experiment=REAL_DATASET_EXPERIMENT,
                    run_metadata=expected_real_metadata,
                )
                if source_labels:
                    print(
                        f"Reused cached real-world W&B result for {model_name} from stored labels "
                        f"{sorted(source_labels)}: {cached_bundle_path}. Saved local alias bundle: {model_bundle_path}"
                    )
                else:
                    print(
                        f"Reused cached real-world W&B result for {model_name}: {cached_bundle_path}. "
                        f"Saved local alias bundle: {model_bundle_path}"
                    )
                continue

    print(f"Running real-world benchmark for model: {model_name}")
    results = run_real_world_model_from_config(
        model_config=model_config,
        experiment=REAL_DATASET_EXPERIMENT,
        device=device,
        baseline_n_jobs=BASELINE_EVAL["n_jobs"],
        random_state=BASELINE_EVAL["random_state"],
        verbose=False,
    )

    if not results.empty:
        results["model"] = model_name
    else:
        print(f"Warning: No results for model {model_name}, skipping saving and upload.")
        continue
    summary = summarize_results(results)
    per_dataset = compute_per_dataset_stats(results)

    model_bundle_path = make_bundle_path(
        OUTPUT_ROOT / "real_world",
        f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_dataframe_bundle(
        dataframes={
            "results": results,
            "summary": summary.reset_index() if summary is not None else None,
            "per_dataset": per_dataset,
        },
        bundle_dir=model_bundle_path,
        experiment=REAL_DATASET_EXPERIMENT,
        run_metadata=expected_real_metadata,
    )
    print(f"Saved real-world bundle for {model_name}: {model_bundle_path}")

    if WANDB["enabled"]:
        artifact_ref = upload_results_bundle_to_wandb(
            model_bundle_path,
            artifact_name=model_artifact_name,
            run_name=f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}_{model_hash}_artifact",
            metadata={
                "experiment": REAL_DATASET_EXPERIMENT,
                "model_name": model_name,
                "model_config": model_config,
                "model_hash": model_hash,
                "run_metadata": expected_real_metadata,
            },
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
            job_type="real_world_bundle_upload",
        )
        print(f"Uploaded real-world artifact for {model_name}: {artifact_ref}")

print("\nReal-world evaluation completed.")


## Load and aggregate per-model outputs

Collect the latest compatible bundle for each selected model, then build combined result tables.

In [ ]:
from IPython.display import display

from pfns.datasets.tabular_datasets import (
    get_benchmark_suite_dids,
    open_cc_dids as OPENCC_BENCHMARK,
    test_dids_classification as TEST_BENCHMARK,
)

bundles_by_model = {}
missing_models = []

for model_name in models_to_compare:
    bundle_path, bundle = find_latest_real_world_bundle_for_model(
        model_name,
        output_root=OUTPUT_ROOT,
        expected_metadata=expected_real_metadata,
    )
    if bundle is None:
        missing_models.append(model_name)
        continue

    bundles_by_model[model_name] = {"path": bundle_path, "bundle": bundle}
    print(f"Loaded {model_name} bundle: {bundle_path}")

if not bundles_by_model:
    raise RuntimeError(
        "No compatible bundles were found. Run the benchmark cell first, or check metadata settings."
    )

all_results, results_by_model = aggregate_real_world_results_from_bundles(
    bundles_by_model,
    expected_splits=int(REAL_DATASET_EXPERIMENT["n_splits"]),
)

summary = summarize_results(all_results)
per_dataset = compute_per_dataset_stats(all_results)
if summary is None or per_dataset is None or per_dataset.empty:
    raise RuntimeError("Could not compute summary tables from aggregated results.")

benchmark = REAL_DATASET_EXPERIMENT["benchmark"]
if benchmark == "opencc":
    benchmark_ids = OPENCC_BENCHMARK
elif benchmark == "test":
    benchmark_ids = TEST_BENCHMARK
elif benchmark == "openml_large_dataset":
    benchmark_ids = [1461]
elif benchmark == "tabarena_full":
    benchmark_ids = get_benchmark_suite_dids()
elif benchmark == "tabarena_medium":
    benchmark_ids = get_benchmark_suite_dids(
        min_samples=10_000,
        max_samples=250_000,
    )
else:
    raise ValueError(f"Unsupported benchmark: {benchmark}")

print("\nAggregated real-world benchmark summary")
print(f"Models loaded: {len(bundles_by_model)} / {len(models_to_compare)}")
print(f"Rows in all_results: {len(all_results)}")
print(f"Datasets covered: {per_dataset['dataset'].nunique()} / {len(benchmark_ids)}")

if missing_models:
    print("Missing compatible bundles for:", missing_models)

summary_metrics = [
    {"metric": "accuracy", "show_std": True},
    {"metric": "roc_auc", "show_std": True},
    #{"metric": "log_loss", "show_std": True},
    #{"metric": "ece", "show_std": True},
    {"metric": "fit_time", "show_std": False},
    {"metric": "predict_time", "show_std": False},
]

print_results_summary(
    all_results,
    title="Aggregated Results Across Selected Models",
    metrics=summary_metrics,
)



## Result tables

Display model-level and dataset-level metric tables from the aggregated benchmark results.

In [ ]:
from pfns.experiments.model_benchmarks.plotting import resolve_display_name_map

if summary is None or per_dataset is None or summary.empty or per_dataset.empty:
    raise RuntimeError("No summary/per-dataset results available.")

display_name_map = resolve_display_name_map(all_results)

def format_model_label(label):
    return display_name_map.get(str(label), str(label))

summary_display = summary.copy().sort_values("accuracy_mean", ascending=False)
summary_display.index = summary_display.index.map(format_model_label)
summary_metric_labels = {
    "accuracy_mean": "Accuracy mean \u2191",
    "accuracy_std": "Accuracy std \u2191",
    "roc_auc_mean": "ROC-AUC mean \u2191",
    "roc_auc_std": "ROC-AUC std \u2191",
    "log_loss_mean": "CE mean \u2193",
    "log_loss_std": "CE std \u2193",
    "ece_mean": "ECE mean \u2193",
    "ece_std": "ECE std \u2193",
}
summary_display = summary_display.rename(columns=summary_metric_labels)
print("Model summary table (mean over datasets):")
display(
    summary_display.style.format(
        {
            "Accuracy mean \u2191": "{:.4f}",
            "Accuracy std \u2191": "{:.4f}",
            "ROC-AUC mean \u2191": "{:.4f}",
            "ROC-AUC std \u2191": "{:.4f}",
            "CE mean \u2193": "{:.4f}",
            "CE std \u2193": "{:.4f}",
            "ECE mean \u2193": "{:.4f}",
            "ECE std \u2193": "{:.4f}",
            "fit_time_mean": "{:.3f}",
            "predict_time_mean": "{:.3f}",
        }
    ).background_gradient(cmap="Greens_r", subset=["Accuracy mean \u2191", "ROC-AUC mean \u2191"])
)

per_dataset_metric_specs = [
    ("accuracy_mean", "Accuracy \u2191"),
    ("roc_auc_mean", "ROC-AUC \u2191"),
    ("log_loss_mean", "CE \u2193"),
    ("ece_mean", "ECE \u2193"),
]
for metric, metric_label in per_dataset_metric_specs:
    print(f"\nPer-dataset table: {metric_label}")
    table = (
        per_dataset
        .pivot_table(index="dataset", columns="model", values=metric, observed=True)
        .sort_index()
    )
    table = table.rename(columns=format_model_label)
    display(table.style.format("{:.4f}").background_gradient(cmap="Blues"))


## Plots

Create metric bars, speed-vs-accuracy tradeoff, and dataset-level rank curves using existing benchmark utilities.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

from pfns.experiments.model_benchmarks.analysis import compute_mean_rank_tables
from pfns.experiments.model_benchmarks.constants import DEFAULT_COLORS
from pfns.experiments.model_benchmarks.plotting import (
    build_model_style_map,
    plot_curves_from_df,
    resolve_display_name_map,
)

if summary is None or summary.empty:
    raise RuntimeError("No summary dataframe available for plotting.")

display_name_map = resolve_display_name_map(all_results)

ordered_models = summary.sort_values("accuracy_mean", ascending=False).index.tolist()
non_black_colors = [c for c in DEFAULT_COLORS if c.lower() not in {"#000000", "black", "k"}]
style_map = build_model_style_map(ordered_models, colors=non_black_colors)
model_colors = {name: style_map[name][2] for name in ordered_models}
metric_labels = {
    "accuracy": "Accuracy \u2191",
    "roc_auc": "ROC-AUC \u2191",
    "log_loss": "CE \u2193",
}

SUMMARY_TITLE_FONTSIZE = 20
SUMMARY_XTICK_FONTSIZE = 15
SUMMARY_YTICK_FONTSIZE = 15

SUMMARY_ERROR_BAR_MODE = None # "std"  # 'std' or None
error_bar_title_suffix = " (std)" if SUMMARY_ERROR_BAR_MODE == "std" else ""
error_col_by_mode = {
    "std": {
        "accuracy": "accuracy_std",
        "roc_auc": "roc_auc_std",
        "log_loss": "log_loss_std",
    },
    None: {
        "accuracy": None,
        "roc_auc": None,
        "log_loss": None,
    },
}
if SUMMARY_ERROR_BAR_MODE not in error_col_by_mode:
    raise ValueError("SUMMARY_ERROR_BAR_MODE must be 'std' or None.")

# 1) Summary metric bar plots
bar_specs = [
    ("accuracy", "accuracy_mean", metric_labels["accuracy"], True),
    ("roc_auc", "roc_auc_mean", metric_labels["roc_auc"], True),
    ("log_loss", "log_loss_mean", metric_labels["log_loss"], False),
]

fig = plt.figure(figsize=(20, 12.5), dpi=400)
grid = fig.add_gridspec(2, 10, hspace=0.20, wspace=0.40)
axes = [
    fig.add_subplot(grid[0, :4]),
    fig.add_subplot(grid[0, 6:]),
    fig.add_subplot(grid[1, 3:7]),
]
for ax, (metric_key, mean_col, title, higher_is_better) in zip(axes, bar_specs):
    plot_df = summary.reset_index().rename(columns={"index": "model"})
    plot_df = plot_df.sort_values(mean_col, ascending=not higher_is_better)
    plot_df["display_model"] = plot_df["model"].map(lambda model: display_name_map.get(str(model), str(model)))

    err_col = error_col_by_mode[SUMMARY_ERROR_BAR_MODE][metric_key]
    err = plot_df[err_col].fillna(0.0) if err_col is not None else None
    ax.barh(
        plot_df["display_model"],
        plot_df[mean_col],
        xerr=err,
        color=[model_colors[m] for m in plot_df["model"]],
        alpha=0.9,
    )

    lower_series = plot_df[mean_col] - err if err is not None else plot_df[mean_col]
    upper_series = plot_df[mean_col] + err if err is not None else plot_df[mean_col]
    lower = float(lower_series.min())
    upper = float(upper_series.max())
    span = max(upper - lower, 1e-6)
    pad = 0.08 * span
    x_min = lower - pad
    x_max = upper + pad

    if mean_col in {"accuracy_mean", "roc_auc_mean"}:
        x_min = max(0.0, x_min)
        x_max = min(1.0, x_max)
    else:
        x_min = max(0.0, x_min)

    if x_max <= x_min:
        x_max = x_min + max(span, 1e-3)

    ax.set_xlim(x_min, x_max)
    
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
    ax.tick_params(axis="x", labelsize=SUMMARY_XTICK_FONTSIZE)
    
    ax.set_title(f"{title}{error_bar_title_suffix}", fontsize=SUMMARY_TITLE_FONTSIZE, pad=10)
    ax.grid(axis="x", alpha=0.25)
    ax.tick_params(axis="y", labelsize=SUMMARY_YTICK_FONTSIZE, labelleft=True, labelright=False, pad=4)

fig.subplots_adjust(left=0.20, right=0.98, top=0.93, bottom=0.08)
plt.show()

# 2) Accuracy vs inference speed tradeoff
tradeoff_df = summary.reset_index().rename(columns={"index": "model"})
tradeoff_df["display_model"] = tradeoff_df["model"].map(
    lambda model: display_name_map.get(str(model), str(model))
)
fig, ax = plt.subplots(figsize=(9, 6), dpi=400)
for _, row in tradeoff_df.iterrows():
    model_name = row["model"]
    display_model_name = row["display_model"]
    ax.scatter(
        row["predict_time_mean"],
        row["accuracy_mean"],
        s=90,
        color=model_colors[model_name],
        alpha=0.9,
        label=display_model_name,
    )
    ax.annotate(display_model_name, (row["predict_time_mean"], row["accuracy_mean"]), xytext=(4, 4), textcoords="offset points", fontsize=9)

ax.set_xlabel("Predict time mean (s)")
ax.set_ylabel(f"{metric_labels['accuracy']} mean")
ax.set_xscale("log")
ax.grid(alpha=0.25)
ax.set_title(f"{metric_labels['accuracy']} vs Prediction Time (s) \u2193")
plt.show()

# 3) Dataset-wise rank curves with model_benchmarks rank + plotting helpers
rank_base = (
    all_results
    .groupby(["model", "dataset"], observed=True)
    .agg(
        accuracy=("accuracy", "mean"),
        roc_auc=("roc_auc", "mean"),
        log_loss=("log_loss", "mean"),
    )
    .reset_index()
)

rank_long = rank_base.melt(
    id_vars=["model", "dataset"],
    value_vars=["accuracy", "roc_auc", "log_loss"],
    var_name="metric",
    value_name="value",
)
rank_long["metric"] = rank_long["metric"].replace({"accuracy": "acc"})

dataset_to_idx = {
    name: idx
    for idx, name in enumerate(sorted(rank_long["dataset"].unique()), start=1)
}
rank_input = rank_long.assign(seqlen=rank_long["dataset"].map(dataset_to_idx))
rank_input["rep"] = rank_input["dataset"]

rank_tables = compute_mean_rank_tables(
    rank_input,
    x_col="seqlen",
    higher_is_better_metrics={"acc", "roc_auc"},
)

overall_ranks = rank_tables["overall_ranks"].copy()
overall_ranks["metric"] = overall_ranks["metric"].replace({"acc": "accuracy"})
rank_metric_labels = {
    "accuracy": "Accuracy Rank \u2193",
    "roc_auc": "ROC-AUC Rank \u2193",
    "log_loss": "CE Rank \u2193",
}
print("Overall mean rank (lower is better \u2193):")
for metric in ["accuracy", "roc_auc", "log_loss"]:
    ranked = (
        overall_ranks[overall_ranks["metric"] == metric]
        .sort_values("rank")
        .drop(columns="metric")
        .reset_index(drop=True)
    )
    ranked["model"] = ranked["model"].map(format_model_label)
    print(f"\n{rank_metric_labels[metric]}")
    display(ranked.style.format({"rank": "{:.2f}"}).background_gradient(subset=["rank"], cmap="Greens_r"))

rank_curve_df = rank_tables["x_ranks"].copy()
rank_curve_df["metric"] = rank_curve_df["metric"].replace(
    {
        "acc": "accuracy_rank",
        "roc_auc": "roc_auc_rank",
        "log_loss": "log_loss_rank",
    }
)

plot_curves_from_df(
    rank_curve_df,
    specs=[
        ("accuracy_rank", "Accuracy Rank \u2193"),
        ("roc_auc_rank", "ROC-AUC Rank \u2193"),
        ("log_loss_rank", "CE Rank \u2193"),
    ],
    style_map=style_map,
    x_col="seqlen",
    value_col="rank",
    x_label="Dataset index (sorted by dataset name)",
    title_suffix=" (1 is best)",
    invert_y=True,
    show_std=False,
    figsize=(24, 7),
)



## Normalized metrics plots

In [ ]:
import seaborn as sns

from pfns.experiments.model_benchmarks.analysis import (
    add_normalized_comparison_metrics,
    add_numeric_buckets,
)

metric_specs = [
    ("normalized_accuracy", "Normalized Accuracy \u2191"),
    ("normalized_roc_auc", "Normalized ROC-AUC \u2191"),
    ("normalized_log_loss", "Normalized CE \u2191"),
]
raw_metric_cols = ["accuracy", "roc_auc", "log_loss"]

def _format_bucket_bound(value: float) -> str:
    value = int(round(float(value)))
    if value >= 1000:
        short = f"{value / 1000:.1f}k"
        return short.replace(".0k", "k")
    return f"{value}"

if "dataset_num_rows" not in all_results.columns:
    raise RuntimeError("all_results does not include dataset_num_rows, so dataset-size analysis cannot be plotted.")

dataset_metric_base = per_dataset.rename(
    columns={
        "dataset_num_rows_first": "dataset_num_rows",
        "accuracy_mean": "accuracy",
        "roc_auc_mean": "roc_auc",
        "log_loss_mean": "log_loss",
    }
)[["model", "dataset", "dataset_num_rows", *raw_metric_cols]].dropna(
    subset=["dataset_num_rows", *raw_metric_cols]
)

normalized_dataset_size_df = add_normalized_comparison_metrics(
    dataset_metric_base.melt(
        id_vars=["model", "dataset", "dataset_num_rows"],
        value_vars=raw_metric_cols,
        var_name="metric",
        value_name="value",
    ).dropna(subset=["dataset_num_rows", "value"]).assign(rep=lambda df: df["dataset"]),
    metric_keys=raw_metric_cols,
    higher_is_better_metrics={"accuracy", "roc_auc"},
    group_cols=("dataset",),
)
normalized_dataset_size_df = add_numeric_buckets(
    normalized_dataset_size_df[
        normalized_dataset_size_df["metric"].isin([metric for metric, _ in metric_specs])
    ].copy(),
    value_col="dataset_num_rows",
    bucket_col="size_bucket",
    q=DATASET_SIZE_BUCKET_Q,
).dropna(subset=["size_bucket"])
bucket_meta = (
    normalized_dataset_size_df.groupby("size_bucket", observed=True)
    .agg(
        min_rows=("dataset_num_rows", "min"),
        max_rows=("dataset_num_rows", "max"),
        n_datasets=("dataset", "nunique"),
    )
    .reset_index()
)
bucket_label_map = {
    row["size_bucket"]: (
        f"{_format_bucket_bound(row['min_rows'])}-{_format_bucket_bound(row['max_rows'])}"
        f"\nn={int(row['n_datasets'])}"
    )
    for _, row in bucket_meta.iterrows()
}
normalized_dataset_size_df["size_bucket"] = normalized_dataset_size_df["size_bucket"].cat.rename_categories(bucket_label_map)
normalized_dataset_size_df["bucket_idx"] = normalized_dataset_size_df["size_bucket"].cat.codes
bucket_order = normalized_dataset_size_df["size_bucket"].cat.categories.tolist()
bucket_summary = (
    normalized_dataset_size_df.groupby(["metric", "model", "size_bucket"], observed=True)["value"]
    .mean()
    .reset_index()
)
model_order = [model_name for model_name in summary.index.tolist() if model_name in set(bucket_summary["model"])]

fig, axes = plt.subplots(
    1,
    len(metric_specs),
    figsize=(24, max(5.0, 0.45 * len(model_order) + 1.5)),
    dpi=300,
    constrained_layout=True,
)
if len(metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, metric_specs):
    heatmap_df = (
        bucket_summary[bucket_summary["metric"] == metric_key]
        .assign(size_bucket=lambda df: df["size_bucket"].astype(str))
        .pivot(index="model", columns="size_bucket", values="value")
        .reindex(index=model_order, columns=bucket_order)
    )
    heatmap_df.index = [format_model_label(model_name) for model_name in heatmap_df.index]
    sns.heatmap(
        heatmap_df,
        ax=ax,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        linecolor="white",
        cbar=ax is axes[-1],
        cbar_kws={"label": "Mean normalized score"},
        annot_kws={"fontsize": 7},
    )
    ax.set_title(metric_title)
    ax.set_xlabel("")
    ax.set_ylabel("Model" if ax is axes[0] else "")
    ax.tick_params(axis="x", labelrotation=0, labelsize=8, pad=6)

plt.show()


In [ ]:
TOP_MODELS_FOR_BUCKET_CI_PLOTS = 8

selected_models = model_order[:TOP_MODELS_FOR_BUCKET_CI_PLOTS]
curve_df = normalized_dataset_size_df[
    normalized_dataset_size_df["model"].isin(selected_models)
].copy()

_, axes = plot_curves_from_df(
    curve_df,
    specs=metric_specs,
    style_map=style_map,
    x_col="bucket_idx",
    value_col="value",
    x_label="Dataset size bucket",
    title_suffix=" by Dataset Size Bucket",
    error_bars="ci95",
    error_style="bars",
    figsize=(24, 7),
)
if axes is not None:
    for ax, (_, metric_title) in zip(axes, metric_specs):
        ax.set_ylabel(metric_title)
        ax.set_xticks(range(len(bucket_order)))
        ax.set_xticklabels(bucket_order, rotation=0, ha="center", fontsize=9)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(axis="y", alpha=0.25)

plt.show()


## Setting Analysis Shared Setup

Shared helpers/constants used by both cross-mode ranking and gain/significance sections.


In [ ]:
from pfns.experiments.model_benchmarks.setting_analysis import (
    SETTING_METRIC_DIRECTION,
    SETTING_METRIC_LABELS,
    get_setting_preprocess,
)
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis
from pfns.experiments.model_benchmarks.plotting import resolve_display_name_map

display_name_map = resolve_display_name_map(all_results)

def format_model_label(label):
    return display_name_map.get(str(label), str(label))



## Cross-Mode Setting Ranking

Rank and average `Comb_MT`, `Comb_ST`, `Int_ST`, `Int_MT` across model types using only canonical model names.

Models with extra suffixes (for example `short_conv`, hidden-size sweeps, `Layers_24`, or seq-len variants) are ignored automatically.


In [ ]:
import pandas as pd

if per_dataset is None or per_dataset.empty:
    raise RuntimeError("No per-dataset results are available for setting ranking.")

TARGET_SETTINGS = ["Comb_MT", "Comb_ST", "Int_ST", "Int_MT"]

pre = get_setting_preprocess(
    results_df=per_dataset,
    target_settings=TARGET_SETTINGS,
)

evaluated_df = pre["model_meta"]
evaluated_presence = pre["presence"].reindex(columns=TARGET_SETTINGS, fill_value=False).sort_index()
evaluated_complete_types = pre["eligible_model_types"]
analysis_df = pre["filtered_results"].copy()

print("Canonical setting availability in currently loaded evaluation results.")
display(evaluated_presence)
print(f"Model types with all four settings evaluated: {evaluated_complete_types}")

metric_specs = [
    ("accuracy_mean", True, "Accuracy"),
    ("roc_auc_mean", True, "ROC-AUC"),
    ("log_loss_mean", False, "CE"),
    ("ece_mean", False, "ECE"),
]

missing_metric_cols = [col for col, _, _ in metric_specs if col not in analysis_df.columns]
if missing_metric_cols:
    raise RuntimeError(f"Missing required metric columns for ranking: {missing_metric_cols}")

rank_frames = []
for metric_col, higher_is_better, metric_label in metric_specs:
    metric_rank_df = analysis_df[["dataset", "model_type", "setting", metric_col]].dropna().copy()
    metric_rank_df["rank"] = metric_rank_df.groupby(
        ["model_type", "dataset"], observed=True
    )[metric_col].rank(method="average", ascending=not higher_is_better)
    metric_rank_df["metric"] = metric_label
    rank_frames.append(metric_rank_df[["metric", "setting", "rank"]])

rank_df = pd.concat(rank_frames, ignore_index=True)
rank_summary = (
    rank_df.groupby(["metric", "setting"], observed=True)
    .agg(mean_rank=("rank", "mean"), std_rank=("rank", "std"), n_pairs=("rank", "size"))
    .reset_index()
)

rank_table = (
    rank_summary
    .pivot(index="setting", columns="metric", values="mean_rank")
    .reindex(TARGET_SETTINGS)
)

print("Mean rank by setting across model types and datasets (lower is better).")
display(rank_table.style.format("{:.3f}").background_gradient(cmap="Greens_r", axis=0))

overall_setting_rank = (
    rank_summary.groupby("setting", observed=True)["mean_rank"]
    .mean()
    .reindex(TARGET_SETTINGS)
    .sort_values()
)
print("Overall mean rank across all metrics (lower is better).")
display(overall_setting_rank.to_frame("overall_mean_rank").style.format({"overall_mean_rank": "{:.3f}"}).background_gradient(cmap="Greens_r"))

setting_metric_means = (
    analysis_df.groupby("setting", observed=True)
    .agg(
        accuracy_mean=("accuracy_mean", "mean"),
        roc_auc_mean=("roc_auc_mean", "mean"),
        log_loss_mean=("log_loss_mean", "mean"),
        ece_mean=("ece_mean", "mean"),
    )
    .reindex(TARGET_SETTINGS)
)
print("Raw metric averages by setting across eligible model types.")
display(setting_metric_means.style.format({
    "accuracy_mean": "{:.4f}",
    "roc_auc_mean": "{:.4f}",
    "log_loss_mean": "{:.4f}",
    "ece_mean": "{:.4f}",
}))



## Gain And Significance Across Settings

Paired gain and interval analysis for `Comb_MT`, `Comb_ST`, `Int_MT`, `Int_ST` (not model-vs-model).

Positive gain always means the compared setting is better than the reference setting.


In [ ]:
import matplotlib.pyplot as plt

SETTING_ANALYSIS = {
    "metric": "accuracy",  # one of: accuracy, roc_auc, log_loss, ece
    "unit": "dataset",  # dataset or split
    "target_labels": ["Comb_MT", "Comb_ST", "Int_MT", "Int_ST"],
    "reference_label": "Int_MT",
    "error_bars": "ci95",  # std or ci95
    "compare_col": "setting",
    "comparison_label": "setting",
    "wilcoxon_alpha": 0.05,
}

if all_results is None or all_results.empty:
    raise RuntimeError("No aggregated all_results dataframe is available for setting CD analysis.")

metric = SETTING_ANALYSIS["metric"]
if metric not in SETTING_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SETTING_METRIC_DIRECTION)}")
if metric not in all_results.columns:
    raise RuntimeError(f"Metric column {metric!r} is not present in all_results.")

SETTING_DISPLAY_NAMES = {
    "Comb_MT": "Combined \n Multi Target",
    "Comb_ST": "Combined \n Single Target",
    "Int_MT": "Interleaved Multi Target",
    "Int_ST": "Interleaved \n Single Target",
}
SETTING_DISPLAY_NAMES_MULTILINE = {
    "Combined Multi Target": "Combined\nMulti Target",
    "Combined Single Target": "Combined\nSingle Target",
    "Interleaved Multi Target": "Interleaved\nMulti Target",
    "Interleaved Single Target": "Interleaved\nSingle Target",
}

unit = SETTING_ANALYSIS["unit"]
target_labels = list(dict.fromkeys(SETTING_ANALYSIS["target_labels"]))
target_display_labels = [SETTING_DISPLAY_NAMES.get(label, label) for label in target_labels]
reference_display_label = SETTING_DISPLAY_NAMES.get(SETTING_ANALYSIS["reference_label"], SETTING_ANALYSIS["reference_label"])

pre = get_setting_preprocess(
    results_df=all_results,
    target_settings=target_labels,
)
presence = pre["presence"].reindex(columns=target_labels, fill_value=False).rename(columns=SETTING_DISPLAY_NAMES)
eligible_model_types = pre["eligible_model_types"]
comparison_results = pre["filtered_results"]
comparison_results = comparison_results.copy()
comparison_results["setting"] = comparison_results["setting"].map(lambda label: SETTING_DISPLAY_NAMES.get(label, label))

print("Eligible model types (all requested settings available):", eligible_model_types)
print("Setting availability by model type:")
display(presence)

higher_is_better = SETTING_METRIC_DIRECTION[metric]
pair_cols = ["model_type", "dataset"] if unit == "dataset" else ["model_type", "dataset", "split"]

analysis = run_comparison_analysis(
    comparison_results=comparison_results,
    metric_col=metric,
    metric_label=SETTING_METRIC_LABELS[metric],
    compare_col=SETTING_ANALYSIS["compare_col"],
    target_labels=target_display_labels,
    pair_cols=pair_cols,
    higher_better=higher_is_better,
    reference_label=reference_display_label,
    unit=unit,
    error_bars=SETTING_ANALYSIS["error_bars"],
    comparison_label=SETTING_ANALYSIS["comparison_label"],
    include_boxplot=True,
    include_pairwise_tables=True,
    include_cd_diagram=True,
    wilcoxon_alpha=SETTING_ANALYSIS["wilcoxon_alpha"],
    empty_error_message=(
        "No complete paired rows found across all requested settings. "
        "Try unit='dataset' or evaluate more models/settings."
    ),
)

print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
print(f"Metric: {SETTING_METRIC_LABELS[metric]} ({metric})")
print(f"Pairing unit: {unit}")
print(f"Reference setting: {analysis['gain']['reference_label']}")
print(f"Comparisons: {analysis['gain']['comparison_labels']}")
print("Setting means:")
display(analysis["gain"]["label_means"].to_frame("mean"))

print("\nPaired gain summary by setting (positive means better than reference setting):")
display(
    analysis["gain"]["gain_summary"].style.format(
        {
            "mean_gain": "{:+.5f}",
            "std_gain": "{:.5f}",
            "sem_gain": "{:.5f}",
            "ci95": "{:.5f}",
            "ci95_low": "{:+.5f}",
            "ci95_high": "{:+.5f}",
            "n_pairs": "{:.0f}",
            "share_pairs_better": "{:.1%}",
        }
    ).background_gradient(subset=["mean_gain"], cmap="RdYlGn")
)

if analysis["pairwise"] is not None:
    print("\nPairwise mean gain matrix (row setting vs column setting, positive means row is better):")
    display(analysis["pairwise"]["pairwise_mean"].style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None))

    print("Pairwise 95% CI half-width matrix:")
    display(analysis["pairwise"]["pairwise_ci95"].style.format("{:.4f}").background_gradient(cmap="Blues", axis=None))

    print("Pairwise significance proxy (95% CI excludes zero):")
    display(analysis["pairwise"]["pairwise_sig"])

for fig_key in ["gain_barh", "gain_boxplot", "wilcoxon_cd"]:
    if fig_key in analysis["figures"]:
        fig, _ = analysis["figures"][fig_key]
        if fig_key == "gain_barh" and fig.axes:
            fig.axes[0].set_title(f"Paired gain vs {reference_display_label} with 95% confidence intervals")
        if fig_key == "wilcoxon_cd" and fig.axes:
            for text in fig.axes[0].texts:
                if text.get_position()[0] > 0.5:
                    label = text.get_text()
                    text.set_text(SETTING_DISPLAY_NAMES_MULTILINE.get(label, label))
        display(fig)
        plt.close(fig)




## Equal-Params Model Wilcoxon-Holm Comparison

Compare only the registry-defined equal-parameter models using paired Wilcoxon tests with Holm correction.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pfns.experiments.model_benchmarks.model_registry import get_models_from_families
from pfns.experiments.model_benchmarks.setting_analysis import (
    SETTING_METRIC_DIRECTION,
    SETTING_METRIC_LABELS,
)
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis
from pfns.experiments.model_benchmarks.wilcoxon_cd_diagram import wilcoxon_holm_from_wide

EQUAL_PARAMS_ANALYSIS = {
    "metric": "accuracy",  # one of: accuracy, roc_auc, log_loss, ece
    "unit": "dataset",  # dataset or split
    "reference_label": None,  # auto-selects best mean metric if None
    "wilcoxon_alpha": 0.05,
}

if all_results is None or all_results.empty:
    raise RuntimeError("No aggregated all_results dataframe is available for equal-params analysis.")

metric = EQUAL_PARAMS_ANALYSIS["metric"]
if metric not in SETTING_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SETTING_METRIC_DIRECTION)}")
if metric not in all_results.columns:
    raise RuntimeError(f"Metric column {metric!r} is not present in all_results.")

unit = EQUAL_PARAMS_ANALYSIS["unit"]
if unit not in {"dataset", "split"}:
    raise ValueError("EQUAL_PARAMS_ANALYSIS['unit'] must be 'dataset' or 'split'.")

registered_equal_models = list(get_models_from_families(["equal_params"]).keys())
available_models = set(all_results["model"].astype(str).unique())
target_models = [model for model in registered_equal_models if model in available_models]

if len(target_models) < 2:
    raise RuntimeError(
        "Need at least two equal-params models in all_results for Wilcoxon-Holm analysis. "
        f"Found {len(target_models)}: {target_models}."
    )

target_display_models = [format_model_label(model) for model in target_models]
comparison_results = all_results[all_results["model"].isin(target_models)].copy()
comparison_results["model"] = comparison_results["model"].map(format_model_label)
if comparison_results.empty:
    raise RuntimeError("No rows found in all_results for equal-params models.")

higher_is_better = SETTING_METRIC_DIRECTION[metric]
model_means = comparison_results.groupby("model", observed=True)[metric].mean().reindex(target_display_models)

reference_label = EQUAL_PARAMS_ANALYSIS["reference_label"]
if reference_label is None:
    reference_label = model_means.idxmax() if higher_is_better else model_means.idxmin()
elif reference_label not in target_models:
    raise RuntimeError(
        f"Configured reference_label={reference_label!r} is not among available equal-params models: {target_models}"
    )
else:
    reference_label = format_model_label(reference_label)

pair_cols = ["dataset"] if unit == "dataset" else ["dataset", "split"]
analysis = run_comparison_analysis(
    comparison_results=comparison_results,
    metric_col=metric,
    metric_label=SETTING_METRIC_LABELS[metric],
    compare_col="model",
    target_labels=target_display_models,
    pair_cols=pair_cols,
    higher_better=higher_is_better,
    reference_label=reference_label,
    unit=unit,
    error_bars="ci95",
    comparison_label="model",
    include_boxplot=False,
    include_pairwise_tables=False,
    include_cd_diagram=True,
    wilcoxon_alpha=EQUAL_PARAMS_ANALYSIS["wilcoxon_alpha"],
    empty_error_message=(
        "No complete paired rows found across equal-params models. "
        "Try unit='dataset' or evaluate more equal-params models."
    ),
)

p_values, mean_ranks, n_pairs = wilcoxon_holm_from_wide(
    metric_wide_complete=analysis["metric_wide_complete"],
    target_labels=analysis["target_labels"],
    higher_better=higher_is_better,
    alpha=EQUAL_PARAMS_ANALYSIS["wilcoxon_alpha"],
)

print(f"Metric: {SETTING_METRIC_LABELS[metric]} ({metric})")
print(f"Pairing unit: {unit}")
print(f"Equal-params models included: {analysis['target_labels']}")
print(f"Reference model (for gain summary): {analysis['gain']['reference_label']}")
print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
print(f"Wilcoxon/Holm paired rows used: {int(n_pairs)}")

print("\nEqual-params model means:")
display(model_means.sort_values(ascending=not higher_is_better).to_frame("mean"))

print("\nPairwise Wilcoxon p-values with Holm significance flag:")
p_value_table = pd.DataFrame(
    p_values,
    columns=["model_a", "model_b", "p_raw", "significant_holm"],
).sort_values("p_raw", ascending=True, kind="stable")
display(
    p_value_table.style.format({"p_raw": "{:.6f}"}).apply(
        lambda col: ["background-color: #c7f9cc" if bool(v) else "" for v in col],
        subset=["significant_holm"],
    )
)

print("Average ranks used in the CD diagram (1 = best):")
rank_table = mean_ranks.reindex(analysis["target_labels"]).to_frame("average_rank")
display(rank_table.sort_values("average_rank").style.format({"average_rank": "{:.3f}"}))

for fig_key in ["gain_barh", "wilcoxon_cd"]:
    if fig_key in analysis["figures"]:
        fig, _ = analysis["figures"][fig_key]
        display(fig)
        plt.close(fig)

## Transformer Masking Model Comparison

Compare transformer masking variants (from the `transformer_masked` registry family) with paired gains and Wilcoxon-Holm significance.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pfns.experiments.model_benchmarks.model_registry import get_models_from_families
from pfns.experiments.model_benchmarks.setting_analysis import (
    SETTING_METRIC_DIRECTION,
    SETTING_METRIC_LABELS,
)
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis
from pfns.experiments.model_benchmarks.wilcoxon_cd_diagram import wilcoxon_holm_from_wide

TRANSFORMER_MASK_ANALYSIS = {
    "metric": "accuracy",  # one of: accuracy, roc_auc, log_loss, ece
    "unit": "dataset",  # dataset or split
    "reference_label": None,  # auto-selects best mean metric if None
    "wilcoxon_alpha": 0.05,
}

if all_results is None or all_results.empty:
    raise RuntimeError("No aggregated all_results dataframe is available for transformer mask analysis.")

metric = TRANSFORMER_MASK_ANALYSIS["metric"]
if metric not in SETTING_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SETTING_METRIC_DIRECTION)}")
if metric not in all_results.columns:
    raise RuntimeError(f"Metric column {metric!r} is not present in all_results.")

unit = TRANSFORMER_MASK_ANALYSIS["unit"]
if unit not in {"dataset", "split"}:
    raise ValueError("TRANSFORMER_MASK_ANALYSIS['unit'] must be 'dataset' or 'split'.")

registered_mask_models = list(get_models_from_families(["transformer_masked"]).keys())
available_models = set(all_results["model"].astype(str).unique())
target_models = [model for model in registered_mask_models if model in available_models]

if len(target_models) < 2:
    print(
        "Skipping transformer mask comparison: need at least two transformer_masked models in all_results. "
        f"Found {len(target_models)}: {target_models}."
    )
    print("Tip: set models_to_compare to include get_models_from_families(['transformer_masked']).")
else:
    target_display_models = [format_model_label(model) for model in target_models]
    comparison_results = all_results[all_results["model"].isin(target_models)].copy()
    comparison_results["model"] = comparison_results["model"].map(format_model_label)
    higher_is_better = SETTING_METRIC_DIRECTION[metric]
    model_means = comparison_results.groupby("model", observed=True)[metric].mean().reindex(target_display_models)

    reference_label = TRANSFORMER_MASK_ANALYSIS["reference_label"]
    if reference_label is None:
        reference_label = model_means.idxmax() if higher_is_better else model_means.idxmin()
    elif reference_label not in target_models:
        raise RuntimeError(
            f"Configured reference_label={reference_label!r} is not among available transformer mask models: {target_models}"
        )
    else:
        reference_label = format_model_label(reference_label)

    pair_cols = ["dataset"] if unit == "dataset" else ["dataset", "split"]
    analysis = run_comparison_analysis(
        comparison_results=comparison_results,
        metric_col=metric,
        metric_label=SETTING_METRIC_LABELS[metric],
        compare_col="model",
        target_labels=target_display_models,
        pair_cols=pair_cols,
        higher_better=higher_is_better,
        reference_label=reference_label,
        unit=unit,
        error_bars="ci95",
        comparison_label="model",
        include_boxplot=False,
        include_pairwise_tables=False,
        include_cd_diagram=True,
        wilcoxon_alpha=TRANSFORMER_MASK_ANALYSIS["wilcoxon_alpha"],
        empty_error_message=(
            "No complete paired rows found across transformer mask models. "
            "Try unit='dataset' or evaluate more transformer_masked models."
        ),
    )

    p_values, mean_ranks, n_pairs = wilcoxon_holm_from_wide(
        metric_wide_complete=analysis["metric_wide_complete"],
        target_labels=analysis["target_labels"],
        higher_better=higher_is_better,
        alpha=TRANSFORMER_MASK_ANALYSIS["wilcoxon_alpha"],
    )

    print(f"Metric: {SETTING_METRIC_LABELS[metric]} ({metric})")
    print(f"Pairing unit: {unit}")
    print(f"Transformer mask models included: {analysis['target_labels']}")
    print(f"Reference model (for gain summary): {analysis['gain']['reference_label']}")
    print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
    print(f"Wilcoxon/Holm paired rows used: {int(n_pairs)}")

    print("\nTransformer mask model means:")
    display(model_means.sort_values(ascending=not higher_is_better).to_frame("mean"))

    print("\nPairwise Wilcoxon p-values with Holm significance flag:")
    p_value_table = pd.DataFrame(
        p_values,
        columns=["model_a", "model_b", "p_raw", "significant_holm"],
    ).sort_values("p_raw", ascending=True, kind="stable")
    display(
        p_value_table.style.format({"p_raw": "{:.6f}"}).apply(
            lambda col: ["background-color: #c7f9cc" if bool(v) else "" for v in col],
            subset=["significant_holm"],
        )
    )

    print("Average ranks used in the CD diagram (1 = best):")
    rank_table = mean_ranks.reindex(analysis["target_labels"]).to_frame("average_rank")
    display(rank_table.sort_values("average_rank").style.format({"average_rank": "{:.3f}"}))

    for fig_key in ["gain_barh", "wilcoxon_cd"]:
        if fig_key in analysis["figures"]:
            fig, _ = analysis["figures"][fig_key]
            display(fig)
            plt.close(fig)

# Model size report for all registry models

In [ ]:
import gc
import os

import pandas as pd
import torch

from pfns.experiments.model_benchmarks.model_registry import get_all_models
from pfns.run_logger import download_model_from_wandb
from pfns.scripts.tabpfn_interface import load_model_workflow

VOCAB_MAPPING_HINTS = ('vocab_mapping', 'vocab_map')

def _is_fla_model(model):
    backbone = getattr(model, 'transformer_layers', None)
    if backbone is None:
        return False
    class_name = backbone.__class__.__name__.lower()
    return class_name == 'flabackbone' or hasattr(backbone, 'fla')

def _is_vocab_mapping_param(name):
    lowered = name.lower()
    return 'fla.embeddings' in lowered

def _load_registry_model(model_config, device):
    base_path = str(model_config.get('base_path') or '.')
    checkpoint_name = str(model_config.get('checkpoint_name') or 'checkpoint.pt')

    wandb_run_id = model_config.get('wandb_run_id')
    if wandb_run_id:
        target_path = download_model_from_wandb(
            wandb_run_id,
            destination_path=model_config.get('destination_path'),
        )
        base_path = os.path.dirname(target_path)
        checkpoint_name = os.path.basename(target_path)

    model, _, _ = load_model_workflow(
        name=checkpoint_name,
        base_path=base_path,
        device=device,
    )
    model.eval()
    return model

def _compute_param_stats(model):
    total_params = 0
    total_bytes = 0
    excluded_params = 0
    excluded_bytes = 0
    excluded_names = []
    is_fla = _is_fla_model(model)

    for name, param in model.named_parameters():
        param_count = int(param.numel())
        param_bytes = param_count * int(param.element_size())
        total_params += param_count
        total_bytes += param_bytes

        if is_fla and _is_vocab_mapping_param(name):
            excluded_params += param_count
            excluded_bytes += param_bytes
            excluded_names.append(name)

    return {
        'is_fla': is_fla,
        'params_total': total_params,
        'params_excluded_vocab_mapping': excluded_params,
        'params_effective': total_params - excluded_params,
        'size_mib_total': total_bytes / (1024 ** 2),
        'size_mib_excluded_vocab_mapping': excluded_bytes / (1024 ** 2),
        'size_mib_effective': (total_bytes - excluded_bytes) / (1024 ** 2),
        'excluded_param_names': ';'.join(excluded_names),
    }

all_models = get_all_models()
report_device = globals().get('device', 'cpu')
rows = []

for idx, (model_name, model_config) in enumerate(sorted(all_models.items()), start=1):
    print(f'[{idx}/{len(all_models)}] Loading {model_name}')
    model = None
    row = {
        'model_name': model_name,
        'display_name': model_config.get('display_name', model_name),
        'wandb_run_id': model_config.get('wandb_run_id', ''),
        'status': 'ok',
        'error': '',
    }
    try:
        model = _load_registry_model(model_config, device=report_device)
        row.update(_compute_param_stats(model))
    except Exception as exc:
        row['status'] = 'error'
        row['error'] = f'{type(exc).__name__}: {exc}'
    finally:
        rows.append(row)
        if model is not None:
            del model
        gc.collect()
        if str(report_device).startswith('cuda') and torch.cuda.is_available():
            torch.cuda.empty_cache()

df_model_sizes = pd.DataFrame(rows).sort_values(by=['status', 'model_name']).reset_index(drop=True)
display_cols = [
    'model_name',
    'display_name',
    'is_fla',
    'params_total',
    'params_excluded_vocab_mapping',
    'params_effective',
    'size_mib_total',
    'size_mib_excluded_vocab_mapping',
    'size_mib_effective',
    'status',
    'error',
]
display(df_model_sizes[display_cols])

## Retrofit legacy checkpoints with input normalization and upload to W&B

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import wandb
from dataclasses import replace
from IPython.display import display

from pfns.run_logger import download_model_from_wandb
from pfns.train import MainConfig
from pfns.utils import strip_compiled_state_dict_prefix

INPUT_NORMALIZATION_RETROFIT = {
    "enabled": True,
    "output_dir": OUTPUT_ROOT / "retrofitted_models",
    "upload_to_wandb": True,
    "wandb_mode": WANDB["mode"],
    "artifact_aliases": ["latest", "input-normalization-retrofit"],
}


def _canonicalize_state_dict_for_model(*, model, source_state_dict):
    normalized_state_dict = dict(source_state_dict)
    normalized_state_dict, _ = strip_compiled_state_dict_prefix(normalized_state_dict)
    model.load_state_dict(dict(normalized_state_dict), strict=True)
    return dict(model.state_dict())


def _remap_state_dict_by_model_structure(*, source_model, target_model, source_state_dict):
    source_model_state = source_model.state_dict()
    target_model_state = target_model.state_dict()
    source_keys = list(source_model_state.keys())
    target_keys = list(target_model_state.keys())

    if len(source_keys) != len(target_keys):
        raise ValueError(
            "Cannot remap checkpoint weights after enabling input normalization: "
            f"source model has {len(source_keys)} state keys, target model has {len(target_keys)}."
        )

    remapped_state = {}
    for source_key, target_key in zip(source_keys, target_keys):
        if source_key not in source_state_dict:
            raise KeyError(
                "Checkpoint is missing expected parameter key "
                f"{source_key!r} while remapping input-normalization override."
            )
        if source_model_state[source_key].shape != target_model_state[target_key].shape:
            raise ValueError(
                "Cannot remap checkpoint weights after enabling input normalization: "
                f"shape mismatch {source_key} {tuple(source_model_state[source_key].shape)} "
                f"-> {target_key} {tuple(target_model_state[target_key].shape)}."
            )
        remapped_state[target_key] = source_state_dict[source_key]

    return remapped_state


def _maybe_enable_input_normalization_encoder(*, config, model_state):
    encoder_config = getattr(config.model, "encoder", None)
    if encoder_config is None or encoder_config.train_normalization:
        return config, model_state, False

    source_model = config.model.create_model()
    canonical_source_state = _canonicalize_state_dict_for_model(
        model=source_model,
        source_state_dict=model_state,
    )
    updated_encoder_config = replace(encoder_config, train_normalization=True)
    updated_model_config = replace(config.model, encoder=updated_encoder_config)
    updated_config = replace(config, model=updated_model_config)
    target_model = updated_config.model.create_model()
    remapped_state = _remap_state_dict_by_model_structure(
        source_model=source_model,
        target_model=target_model,
        source_state_dict=canonical_source_state,
    )
    return updated_config, remapped_state, True

models_to_compare = {
    model_name: model_config
    for model_name, model_config in get_all_models().items()
    if model_config.get("wandb_run_id") is not None
}

def _load_wandb_checkpoint_for_model(model_name, model_config):
    wandb_run_id = model_config.get("wandb_run_id")
    if not wandb_run_id:
        raise ValueError(
            f"{model_name} has no wandb_run_id. This retrofit cell expects W&B-backed checkpoints."
        )

    destination_path = (
        INPUT_NORMALIZATION_RETROFIT["output_dir"]
        / sanitize_wandb_artifact_component(model_name)
        / "checkpoint.pt"
    )
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint_path = download_model_from_wandb(
        wandb_run_id,
        destination_path=str(destination_path),
    )
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    return wandb_run_id, Path(checkpoint_path), checkpoint


def _get_latest_model_artifact_name(run_path):
    api = wandb.Api()
    run = api.run(run_path)
    for artifact in run.logged_artifacts():
        if artifact.type == "model" and "latest" in artifact.aliases:
            return run, artifact.name.split(":", 1)[0]
    raise FileNotFoundError(f"No latest model artifact found for run {run_path}.")


def _replace_model_artifact(*, model_name, source_run_path, checkpoint_path, updated_config):
    source_run, artifact_name = _get_latest_model_artifact_name(source_run_path)
    retrofit_run = wandb.init(
        project=source_run.project,
        entity=source_run.entity,
        id=source_run.id,
        resume="must",
        mode=INPUT_NORMALIZATION_RETROFIT["wandb_mode"],
        job_type="checkpoint_retrofit",
        name=source_run.name,
        config={
            "model_name": model_name,
            "source_run_path": source_run_path,
            "retrofitted_train_normalization": True,
        },
    )

    artifact = wandb.Artifact(
        name=artifact_name,
        type="model",
        description="Checkpoint retrofitted to enable train-time x-encoder normalization.",
        metadata={
            "model_name": model_name,
            "source_run_path": source_run_path,
            "train_normalization": getattr(updated_config.model.encoder, "train_normalization", None),
        },
    )
    artifact.add_file(str(checkpoint_path), name="checkpoint.pt")
    retrofit_run.log_artifact(
        artifact,
        aliases=list(INPUT_NORMALIZATION_RETROFIT["artifact_aliases"]),
    )
    retrofit_run.finish()
    return f"{source_run.entity}/{source_run.project}/{source_run.id}", artifact_name


if not INPUT_NORMALIZATION_RETROFIT["enabled"]:
    print(
        "Set INPUT_NORMALIZATION_RETROFIT['enabled'] = True to rewrite the selected checkpoints and republish them to W&B."
    )
else:
    INPUT_NORMALIZATION_RETROFIT["output_dir"].mkdir(parents=True, exist_ok=True)
    retrofit_rows = []

    for model_name, model_config in models_to_compare.items():
        try:
            source_run_path, checkpoint_path, checkpoint = _load_wandb_checkpoint_for_model(
                model_name,
                model_config,
            )
            config = MainConfig.from_dict(checkpoint["config"])
            updated_config, remapped_state, changed = _maybe_enable_input_normalization_encoder(
                config=config,
                model_state=checkpoint["model_state_dict"],
            )

            status = "already_enabled"
            artifact_name = ""
            if changed:
                patched_checkpoint = dict(checkpoint)
                patched_checkpoint["config"] = updated_config.to_dict()
                patched_checkpoint["model_state_dict"] = remapped_state
                torch.save(patched_checkpoint, checkpoint_path)
                status = "retrofitted"

                print(f"{model_name}: local patched checkpoint -> {checkpoint_path}")

                if INPUT_NORMALIZATION_RETROFIT["upload_to_wandb"]:
                    _, artifact_name = _replace_model_artifact(
                        model_name=model_name,
                        source_run_path=source_run_path,
                        checkpoint_path=checkpoint_path,
                        updated_config=updated_config,
                    )
                    status = "uploaded"
                    print(f"{model_name}: replaced latest model artifact {artifact_name} on {source_run_path}")

            retrofit_rows.append(
                {
                    "model_name": model_name,
                    "status": status,
                    "wandb_run_id": source_run_path,
                    "checkpoint_path": str(checkpoint_path),
                    "artifact_name": artifact_name,
                    "train_normalization": getattr(updated_config.model.encoder, "train_normalization", None),
                    "error": "",
                }
            )
        except Exception as exc:
            retrofit_rows.append(
                {
                    "model_name": model_name,
                    "status": "error",
                    "wandb_run_id": model_config.get("wandb_run_id", ""),
                    "checkpoint_path": "",
                    "artifact_name": "",
                    "train_normalization": None,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            print(f"{model_name}: error -> {type(exc).__name__}: {exc}")

    retrofit_df = pd.DataFrame(retrofit_rows)
    display(retrofit_df)
    print("wandb_run_id stays unchanged for successful rows.")